# Wand-Check mit 3D-Ansicht
BXMid03 · HS26 · Fortgeschritten · Community-Widget ifc-anywidget (MIT) mit IFC-Lite-Renderer

Oben **Modell wählen** (Dropdown), dann *Laufzeit → Alle ausführen*. Eigene Modelle nur hochladen, wenn sie das Projekt verlassen dürfen: Colab läuft auf Google-Servern.

In [ ]:
!pip install -q ifcopenshell ifc-anywidget

In [ ]:
#@title 1 · Modell wählen { run: "auto" }
modell = "Team 5 · Wände (BIMcollab STR, IFC2X3)" #@param ["Team 1 · Brücke (IFC4X3)", "Team 4 · Baufortschritt (Snowdon, IFC2X3)", "Team 5 · Wände (BIMcollab STR, IFC2X3)", "Demo · Holzrahmen fehlerhaft (IFC4)", "Demo · Holzrahmen korrigiert (IFC4)", "Eigene Datei hochladen"]
import urllib.request
BASIS = "https://raw.githubusercontent.com/LTplus-AG/bxmid03-hs26/main/modelle/"
DATEIEN = {
    "Team 1 · Brücke (IFC4X3)": "Team_1_Bruecke_Arbeitsmodell.ifc",
    "Team 4 · Baufortschritt (Snowdon, IFC2X3)": "Team_4_Snowdon_Structural_Baufortschritt.ifc",
    "Team 5 · Wände (BIMcollab STR, IFC2X3)": "Team_5_BIMcollab_Example_STR.ifc",
    "Demo · Holzrahmen fehlerhaft (IFC4)": "Demo_Holzrahmen_fehlerhaft.ifc",
    "Demo · Holzrahmen korrigiert (IFC4)": "Demo_Holzrahmen_korrigiert.ifc",
}
if modell == "Eigene Datei hochladen":
    from google.colab import files
    pfad = list(files.upload())[0]             # Achtung: Datei geht auf Google-Server
else:
    pfad = DATEIEN[modell]
    urllib.request.urlretrieve(BASIS + pfad, pfad)
print("Modell geladen:", pfad)

---
### Nur für Teams mit eigenem Repository
Wer sein Modell schon im Team-Repository hat, fügt hier den GitHub-Link zur IFC-Datei ein. Alle anderen lassen das Feld leer und nutzen die Auswahl oben.

In [ ]:
#@title 1b · Optional: Modell aus dem Team-Repository (überschreibt die Auswahl oben)
github_link = "" #@param {type:"string"}
# Beispiel: https://github.com/LTplus-AG/bxmid03-hs26-team-4/blob/main/model/Modell.ifc
import urllib.parse
if github_link.strip():
    roh = github_link.strip().replace("https://github.com/", "https://raw.githubusercontent.com/").replace("/blob/", "/", 1)
    pfad = urllib.parse.unquote(roh.rsplit("/", 1)[-1])
    urllib.request.urlretrieve(roh, pfad)
    print("Modell aus Team-Repository geladen:", pfad)
else:
    print("Kein Link eingetragen: es gilt die Auswahl oben →", pfad)

---

In [ ]:
#@title 2 · Wand-Check mit IfcOpenShell: Breite > 0
import csv, ifcopenshell
import ifcopenshell.util.element as ue, ifcopenshell.util.unit as uu
m = ifcopenshell.open(pfad)
f = uu.calculate_unit_scale(m)                 # mm -> m
rows = [["GlobalId", "Name", "Breite_m", "Ergebnis"]]
for w in m.by_type("IfcWall"):                 # inkl. IfcWallStandardCase
    q = ue.get_psets(w, qtos_only=True)
    qto = q.get("Qto_WallBaseQuantities") or q.get("BaseQuantities") or {}
    b = qto.get("Width")
    if b is None: erg, ist = "fehlend", None
    else: erg, ist = ("bestanden" if b * f > 0 else "durchgefallen"), b * f
    rows.append([w.GlobalId, w.Name, ist, erg])
with open("wand_breite.csv", "w", newline="") as fh:
    csv.writer(fh).writerows(rows)
from collections import Counter
print(len(rows) - 1, "Wände geprüft:", dict(Counter(r[3] for r in rows[1:])))

In [ ]:
#@title 3 · 3D-Ansicht (Browser mit WebGPU: Chrome oder Edge)
from google.colab import output
output.enable_custom_widget_manager()
from ifc_anywidget import IfcViewer
viewer = IfcViewer(ifc_model=m)
viewer

Im 3D-Fenster eine Wand anklicken, dann die nächste Zelle ausführen.

In [ ]:
#@title 4 · Angeklickte Bauteile mit Prüfergebnis
for gid in viewer.selected_guids or []:
    print(next((r for r in rows if r[0] == gid), [gid, m.by_guid(gid).is_a(), "keine Wand"]))